# A3 CrossGate 诊断复核

门控与同行响应存在，但MG3日末补电及其他微网成本抵消了大部分收益。此笔记本为报告的核算附件。完整状态重建与梯度计算见同目录 analyze.py；报告图表SQL见 report_queries.py。仅seed=42，不作因果归因。

## 1. 读取已执行的诊断与原始来源

输入为同目录 evidence.json，包含原始运行路径、检查点哈希、状态重建误差及完整口径。

In [1]:
from pathlib import Path
import json
import pandas as pd
e=json.loads(Path('evidence.json').read_text(encoding='utf-8'))
assert e['paired']['wins']+e['paired']['losses']==55
for run in e['runs'].values():
    assert run['reconstruction_max_error']<1e-5
print(e['paired'])
print(e['source_files'])

{'mean': 8.302305486843741, 'std': 39.59088394089209, 'median': 12.958609904359719, 'wins': 34, 'losses': 21, 'total': 456.6268017764057}
['outputs\\matd3_plain_single_dispatch\\runs\\MATD3PLAINA3_2026-09-04_16-19-49_seed42\\test_daily_metrics.csv', 'outputs\\matd3_plain_single_dispatch\\runs\\MATD3PLAINA3_2026-09-04_16-19-49_seed42\\training_history.json', 'outputs\\matd3_plain_single_dispatch\\runs\\MATD3PLAINA3_2026-09-04_16-19-49_seed42\\validation_history.json', 'outputs\\matd3_plain_single_dispatch\\runs\\MATD3PLAINA3_2026-09-04_16-19-49_seed42\\actor_diagnostics.csv', 'outputs\\matd3_plain_single_dispatch\\runs\\MATD3PLAINA3_2026-09-04_16-19-49_seed42\\config_snapshot.yaml', 'outputs\\crossgate_single_dispatch\\runs\\CROSSGATESINGLEDISPATCH_2026-09-04_14-03-59_seed42\\test_daily_metrics.csv', 'outputs\\crossgate_single_dispatch\\runs\\CROSSGATESINGLEDISPATCH_2026-09-04_14-03-59_seed42\\training_history.json', 'outputs\\crossgate_single_dispatch\\runs\\CROSSGATESINGLEDISPATCH_202

## 2. 核对经济分解

转账抵消后，各项之和必须等于CrossGate减普通MATD3的日均收益差。

In [2]:
assert abs(sum(r['gain'] for r in e['bridge'])-e['paired']['mean'])<1e-7
assert abs(sum(r['external_profit_gain'] for r in e['microgrid_external'])-e['paired']['mean'])<1e-7
display(pd.DataFrame(e['bridge']))
display(pd.DataFrame(e['microgrid_external']))

,component,plain,crossgate,gain
0,grid_import_cost,1239.554799,1227.666170,11.888629
1,grid_export_revenue,244.871070,240.666850,-4.204220
2,battery_degradation_cost,14.070191,13.496273,0.573918
3,other_cost,2.894923,2.850945,0.043978


,mg,physical_profit_gain,internal_transfer_gain,external_profit_gain
0,mg1,20.004945,-13.680260,33.685206
1,mg2,-1.785997,7.215753,-9.001750
2,mg3,-9.916643,6.464507,-16.381150


## 3. 门控与局部敏感度

敏感度为mean(|du/dx|)乘输入样本标准差，组内求和归一化；不是因果重要性或收益贡献。

In [3]:
display(pd.DataFrame(e['gates']))
display(pd.DataFrame([{'algorithm':r['algorithm'],'microgrid':r['microgrid'],**r['groups']} for r in e['sensitivity']]))

,microgrid,gate,mean,across_states_std_mean,low_rate,high_rate,multiplier_abs_deviation_mean,multiplier_min,multiplier_max
0,mg1,local_gate,0.530464,0.066167,0.000000,0.025189,0.087264,0.820178,1.235637
1,mg1,peer_gate,0.655133,0.084017,0.000000,0.068809,0.104124,0.830225,1.236019
2,mg2,local_gate,0.571561,0.084213,0.000024,0.022206,0.076664,0.797796,1.235702
3,mg2,peer_gate,0.562042,0.086690,0.003196,0.027190,0.091354,0.785849,1.227642
4,mg3,local_gate,0.559191,0.083961,0.012536,0.006132,0.086860,0.761090,1.222120
5,mg3,peer_gate,0.630032,0.091324,0.000521,0.070384,0.107251,0.793038,1.238518


,algorithm,microgrid,local,market,peer
0,matd3_plain_single_dispatch,mg1,0.588104,0.084074,0.327822
1,crossgate_single_dispatch,mg1,0.486056,0.068686,0.445258
2,matd3_plain_single_dispatch,mg2,0.629020,0.071511,0.299469
3,crossgate_single_dispatch,mg2,0.492617,0.063786,0.443597
4,matd3_plain_single_dispatch,mg3,0.597775,0.084097,0.318129
5,crossgate_single_dispatch,mg3,0.507142,0.087431,0.405426


## 4. 日末调度与训练局限

23点补电成本的会计差异不能单独证明某个门控参数造成错误决策。两组均为验证选出的最佳检查点。

In [4]:
display(pd.DataFrame(e['mg3_terminal']))
for x in e['limitations']: print(x)

,algorithm,hour22_soc_mean,hour23_charge_mean,hour23_grid_cost_mean,hour23_forced_rate
0,matd3_plain_single_dispatch,0.402349,15.991392,60.661097,1.0
1,crossgate_single_dispatch,0.146940,30.519267,80.374293,1.0


One training seed; 55 adjacent/seasonal held-out days are not 55 independent training replications.
Parameter gradients depend on parameterization and are not feature importance.
Input sensitivity is a local derivative on recorded-state union, not a causal ablation.
Recorded-action reconstruction verified current data/code against saved profit, SOC, discharge and actor outputs. No training or formal evaluation artifacts were changed.
